In [1]:
import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "torch"], check=True)
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPTokenizer, CLIPTextModel
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

device = "cuda" if torch.cuda.is_available() else "cpu"
log.info(f"Device: {device}")


In [2]:
CSV_PATH   = "/content/cleaned_political_bias_data (1).csv"
CLIP_MODEL = "openai/clip-vit-base-patch32"
BATCH_SIZE = 32
SAVE_PATH  = "/content/news_features.npy"

INPUT_DIM  = 768
HIDDEN_DIM = [512, 256]
OUTPUT_DIM = 128
DROPOUT    = 0.5


In [3]:
CLIP_PROMPT_TEMPLATE = "a photo of v* {text}"

def build_clip_prompt(text: str, max_words: int = 60) -> str:
    words = text.strip().split()[:max_words]
    return CLIP_PROMPT_TEMPLATE.format(text=" ".join(words))

In [4]:
LABEL_MAP   = {"liberal": 0, "center": 1, "conservative": 2}
LABEL_NAMES = ["liberal", "center", "conservative"]

class NewsDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=77, body_words=50):
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=["Title", "News_Body", "Label"])
        self.df         = df[df["Label"].isin(LABEL_MAP)].reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.body_words = body_words

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        title = str(row["Title"]).strip()
        body  = " ".join(str(row["News_Body"]).split()[:self.body_words])
        text  = build_clip_prompt(f"{title}. {body}")
        tokens = self.tokenizer(
            text, max_length=self.max_length,
            padding="max_length", truncation=True, return_tensors="pt"
        )
        return {
            "input_ids":      tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "label":          torch.tensor(LABEL_MAP[row["Label"]], dtype=torch.long),
            "news_id":        torch.tensor(idx, dtype=torch.long),
        }


In [5]:
class LinearProjectionBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.5):
        super().__init__()
        layers = []
        in_dim = input_dim

        for h in hidden_dim:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(p=dropout_rate))
            in_dim = h

        layers.append(nn.Linear(in_dim, output_dim))
        layers.append(nn.BatchNorm1d(output_dim))
        # last layer e relu nei

        self.projection = nn.Sequential(*layers)

    def forward(self, x):
        return self.projection(x)

In [6]:
class NewsFeatureExtractor(nn.Module):
    def __init__(
        self,
        clip_model_name = CLIP_MODEL,
        input_dim       = INPUT_DIM,
        hidden_dim      = HIDDEN_DIM,
        output_dim      = OUTPUT_DIM,
        dropout_rate    = DROPOUT,
        freeze_clip     = True,
    ):
        super().__init__()

        log.info(f"Loading CLIP: {clip_model_name}")
        self.clip_encoder = CLIPTextModel.from_pretrained(clip_model_name)

        if freeze_clip:
            for p in self.clip_encoder.parameters():
                p.requires_grad_(False)
            log.info("CLIP encoder frozen")

        clip_hidden  = self.clip_encoder.config.hidden_size  # 512
        self.adapter = nn.Linear(clip_hidden, input_dim) if clip_hidden != input_dim else nn.Identity()
        log.info(f"Adapter: {clip_hidden} -> {input_dim}")

        self.projection = LinearProjectionBlock(
            input_dim    = input_dim,
            hidden_dim   = hidden_dim,
            output_dim   = output_dim,
            dropout_rate = dropout_rate,
        )
        log.info(f"Projection: {input_dim} -> {hidden_dim} -> {output_dim}")

    def forward(self, input_ids, attention_mask):
        clip_out     = self.clip_encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled       = clip_out.pooler_output       # (B, 512)
        adapted      = self.adapter(pooled)         # (B, 768)
        news_feature = self.projection(adapted)     # (B, output_dim)
        return news_feature


In [7]:
tokenizer = CLIPTokenizer.from_pretrained(CLIP_MODEL)
ds        = NewsDataset(CSV_PATH, tokenizer)
batch     = next(iter(DataLoader(ds, batch_size=4)))

extractor = NewsFeatureExtractor()
extractor.eval()
with torch.no_grad():
    feats = extractor(batch["input_ids"], batch["attention_mask"])

log.info(f"input_ids shape    : {batch['input_ids'].shape}")
log.info(f"After CLIP pooler  : (4, 512)")
log.info(f"After adapter      : (4, {INPUT_DIM})")
log.info(f"News feature shape : {feats.shape}")
log.info("Shape check passed!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_projection.weight                                         | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.

In [8]:
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

extractor = NewsFeatureExtractor().to(device)
extractor.eval()

all_features, all_labels, all_ids = [], [], []

with torch.no_grad():
    for i, batch in enumerate(dl):
        ids   = batch["input_ids"].to(device)
        mask  = batch["attention_mask"].to(device)
        feats = extractor(ids, mask)

        all_features.append(feats.cpu().numpy())
        all_labels.append(batch["label"].numpy())
        all_ids.append(batch["news_id"].numpy())

        if (i + 1) % 20 == 0:
            log.info(f"  Processed {min((i+1)*BATCH_SIZE, len(ds))}/{len(ds)} articles")

features_np = np.vstack(all_features)
labels_np   = np.concatenate(all_labels)
ids_np      = np.concatenate(all_ids)

log.info(f"Extraction complete! Feature matrix: {features_np.shape}")

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_projection.weight                                         | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.

In [9]:
np.save(SAVE_PATH, features_np)
np.save(SAVE_PATH.replace(".npy", "_labels.npy"), labels_np)
np.save(SAVE_PATH.replace(".npy", "_ids.npy"),    ids_np)

log.info(f"Saved: {SAVE_PATH}")
log.info(f"Saved: {SAVE_PATH.replace('.npy', '_labels.npy')}")
log.info(f"Saved: {SAVE_PATH.replace('.npy', '_ids.npy')}")

print("\n── Summary ──")
print(f"Shape  : {features_np.shape}")
print(f"Min    : {features_np.min():.4f}")
print(f"Max    : {features_np.max():.4f}")
print(f"Mean   : {features_np.mean():.4f}")
unique, counts = np.unique(labels_np, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {LABEL_NAMES[u]:>12} : {c} samples")


── Summary ──
Shape  : (9189, 128)
Min    : -0.3045
Max    : 0.3040
Mean   : 0.0046
       liberal : 3065 samples
        center : 3062 samples
  conservative : 3062 samples
